In [39]:
import glob
import json
import os

from dotenv import load_dotenv

import cv2
import matplotlib.pyplot as plt
import numpy as np

In [36]:
images_folder = "/mnt/h/Pronghorn Vertical Imagery/2024/PR527"
research_project = "pronghorn-survey"
model_name = "10-25-2024-16-50-17"

image_files = sorted(glob.glob(os.path.join(images_folder, f"*.[jJ][pP][gG]")))
print(f"{len(image_files)} files found.")

output_folder = os.path.join("/home/koger/pronghorn-processing", model_name)

5722 files found.


## Get names of training images

In [49]:
# Assumes we have a local .env file that stores things like ROOT
load_dotenv()

root = os.environ.get("ROOT")
research_project = "pronghorn-survey"

train_json_path = os.path.join(root, "annotations", research_project, "train.json")

with open(train_json_path) as f:
    train_json = json.load(f)
training_image_names = []
for image_info in train_json['images']:
    training_image_names.append(os.path.splitext(image_info['file_name'])[0])

## Load model predictions

In [10]:
predictions = []
for image_file in image_files:
    image_name = os.path.splitext(os.path.basename(image_file))[0]
    # Get corresponding box file for image
    box_file = os.path.join(output_folder, f"{image_name}_boxes.npy")
    if not os.path.exists(box_file):
        print(f"{image_name} missing box file.")
        boxes = None
    else:
        boxes = np.load(box_file)
    # Get corresponding score file for image
    score_file = os.path.join(output_folder, f"{image_name}_scores.npy")
    if not os.path.exists(score_file):
        print(f"{image_name} missing score file.")
        scores = None
    else:
        scores = np.load(score_file)
    # Get corresponding object class file for image
    label_file = os.path.join(output_folder, f"{image_name}_labels.npy")
    if not os.path.exists(label_file):
        print(f"{image_name} missing labels file.")
        labels = None
    else:
        labels = np.load(label_file)
    image_info = {"image_file": image_file,
                  "boxes": boxes,
                  "scores": scores,
                  "labels": labels
                 }
    predictions.append(image_info)
    

## Calculated highest confidence pronghorn in each image

In [95]:
pronghorn_class = 2
count = 0
for pred in predictions[800:900]:
    if len(pred['labels']) == 0:
        pred['max_pronghorn_score'] = -1
        continue
    is_pronghorn = pred['labels'] == pronghorn_class
    if not np.any(is_pronghorn):
        pred['max_pronghorn_score'] = -1
        continue
    max_score = np.max(pred['scores'][is_pronghorn])
    pred['max_pronghorn_score'] = max_score
    if max_score > .7:
        count += 1
print(f"{count} images with pronghorn above min confidence score." )

# Sort images based on max pronghorn score
predictions = sorted(predictions, key=lambda x: x['max_pronghorn_score'], reverse=True)

100 images with pronghorn above min confidence score.


## Visualize most confident predictions

In [96]:
def is_in_training_set(image_name, training_names):
    """ Check is image name is origin of one of the training images. 
    
    Args:
        image_name: name of target image with extension removed
        training_names: list of all image names in the training set
        
    Returns true is image_name is origin of one of the training images.
    """
    for training_name in training_names:
        if image_name in training_name:
            return True
    return False

save_folder = "/mnt/c/Users/benko/Dropbox/UWyo/pronghorn-census/outputs/top-800-900-0.1"
os.makedirs(save_folder, exist_ok=True)

min_score = .7

draw_box = True
crop_buffer = 100

for pred_num, pred in enumerate(predictions[800:900]):
    image_name = os.path.splitext(os.path.basename(pred['image_file']))[0]
    in_training = is_in_training_set(image_name, training_image_names)
    if in_training:
        print(f"{image_name} is in the training set.")
    image = plt.imread(pred['image_file']).copy()
    
    num_drawn = 0

    crops = []
    scores = []
    
    for box, score, label in zip(pred['boxes'], pred['scores'], pred['labels']):
        if (score < min_score) or (label != pronghorn_class):
            continue
        ymin = np.max([box[1] - crop_buffer, 0])
        ymax = np.min([box[3] + crop_buffer, image.shape[0]])
        xmin = np.max([box[0] - crop_buffer, 0])
        xmax = np.min([box[2] + crop_buffer, image.shape[1]])
        crops.append(image[ymin:ymax, xmin:xmax].copy())
        scores.append(score)
        if draw_box:
            cv2.rectangle(image, (box[0], box[1]), (box[2], box[3]), (255, 0, 0), 3)
        
    fig, axs = plt.subplots(1, len(crops), figsize=(10,3))
    if len(crops) == 1:
        axs = [axs]
    for ax, crop, score in zip(axs, crops, scores):
        ax.imshow(crop)
        ax.set_title(f"score: {score}")
        ax.set_axis_off() 
    fig.suptitle(f"In training set: {in_training}")

    fname = os.path.join(save_folder, f"rank-{pred_num+1:03d}-{image_name}")
    fig.savefig(f"{fname}_crops.jpg", dpi=300, bbox_inches='tight')
    plt.close(fig)
              
    plt.figure(figsize=(15,15))
    plt.imshow(image)
    plt.title(f"In training set: {in_training}")
    plt.axis('off')
    plt.savefig(f"{fname}_whole_image.jpg", dpi=300, bbox_inches='tight')
    plt.close()
    